# 01 Data Collection

## 文献综述

15分钟城市强调居民在日常生活中能够以步行、骑行或公共交通在短时间内获得教育、医疗、商业、休闲与公共服务。这一框架最初被用于重构高密度城市中的生活圈单元，近年来逐渐与健康城市、低碳转型和公平规划议题结合。在健康生活方式研究中，可达性不是单纯的距离问题，而是体育设施、绿地、食品环境、街道网络和社会分层共同作用的结果。国际上有关“food desert”“play desert”“sports desert”的研究普遍指出，弱势群体往往面对更差的公共运动空间、更少的高质量健身设施、更低的绿化水平以及更高的环境暴露风险。与此同时，仅依赖单一设施数量会低估空间结构影响，因此越来越多研究采用多源POI、网络可达性、遥感影像和规则化网格作为综合评价框架。

在方法层面，基于路网的等时圈分析被视为衡量15分钟生活圈的核心技术，因为它比欧氏距离更能反映真实出行阻力；但在超大城市尺度上，全量路网批处理成本极高，因此研究通常采用“局部真实路网 + 全市批量近似阈值”的折中策略，以兼顾精度和可复现性。空间聚合方面，规则方格便于缓存和统计，而Uber H3六边形由于尺度统一、邻接关系稳定、适合多平台发布，已成为城市分析和公众地图中的重要载体。对于健康赛道而言，体育设施、户外健身、公园绿地、健康食品零售和公共游泳池能够共同刻画居民维持主动生活方式的基础条件。若这些资源高度集中在内环核心区而在外围地区形成连续低值带，就会出现典型的“运动荒漠”。

上海是检验这一框架的理想案例。一方面，上海具有高密度中心城区与外围新城并存的空间结构，不同圈层在公共服务供给、土地价值和慢行环境上差异显著；另一方面，本地可获得的多源数据较丰富，包括POI、行政区边界和AI解译的绿地数据，为构建从原始数据到H3评分再到公众GeoJSON发布的端到端流程提供了良好基础。基于此，本组将采用500米网格作为中间分析单元，以四种出行方式的15分钟可达阈值近似等时圈，叠加基础层与Track A健康生活方式赛道指标，在网格尺度完成评分，并进一步聚合到H3六边形，最终输出GeoJSON与交互地图。该流程既保留了研究解释性，也适合课程提交和复现实验。

环境健康和公众展示两个维度被纳入同一套数据链条。AQI日均值能够补充传统POI无法反映的空气暴露差异，使健康生活方式评价不只关注“有没有设施”，也关注“到达设施时是否处在较健康的环境中”。AI解译建筑图层则用于构造陆地掩膜，避免行政边界中的海域、江面和无居住活动空间进入H3评分，从而减少运动荒漠地图的系统性误判。房价与地铁距离并不直接决定健康生活方式质量，但它们可以帮助解释资源分布是否与居住成本、轨道交通便利性产生耦合。由此形成的数据链条从采集、清洗、验证到缓存和发布保持一致，既可服务规划分析，也便于公众理解每个六边形评分背后的数据来源与局限性。


## 目标

本 notebook 完成原始数据盘点、清洗、字段统一和缓存导出。结果将写入 `submit/cache/`，供后续网格、等时圈近似和 H3 评分 notebook 直接复用。


In [ ]:
from pathlib import Path
import json
import re

import geopandas as gpd
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

SUBMIT_DIR = ROOT / "submit"
CACHE_DIR = SUBMIT_DIR / "cache"
OUTPUT_DIR = SUBMIT_DIR / "output"
for folder in [SUBMIT_DIR, CACHE_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_DIR = ROOT / "data"
POI_DIR = DATA_DIR / "POI 2024" / "csv格式" / "已分类"
ADMIN_DIR = DATA_DIR / "03-行政区" / "2024"
AI_DIR = DATA_DIR / "10-AI解译数据"

print("ROOT =", ROOT)
print("CACHE_DIR =", CACHE_DIR)


In [ ]:
raw_inventory = sorted(str(p.relative_to(ROOT)) for p in DATA_DIR.rglob("*") if p.is_file())
pd.DataFrame({"file": raw_inventory}).head(20)


In [ ]:
district_path = ADMIN_DIR / "上海市_县界.shp"
town_path = ADMIN_DIR / "上海市_乡镇边界.shp"

districts = gpd.read_file(district_path).to_crs("EPSG:4326")
towns = gpd.read_file(town_path).to_crs("EPSG:4326")

districts = districts.rename(columns={"县级": "district_name", "县级码": "district_code"})
towns = towns.rename(columns={"name": "town_name", "area_code": "town_code"})

print(districts[["district_name", "district_code"]].head())
print(towns[["town_name", "town_code"]].head())


In [ ]:
boundary_cache = CACHE_DIR / "boundaries.gpkg"
districts[["district_name", "district_code", "geometry"]].to_file(boundary_cache, layer="districts", driver="GPKG")
towns[["town_name", "town_code", "geometry"]].to_file(boundary_cache, layer="towns", driver="GPKG")
boundary_cache


In [ ]:
BASE_RULES = {
    "education": {
        "theme": "base",
        "label": "教育",
        "files": ["上海市-1754933-utf8.csv-科教文化服务.csv"],
        "include": r".*",
        "exclude": None,
    },
    "healthcare": {
        "theme": "base",
        "label": "医疗",
        "files": ["上海市-1754933-utf8.csv-医疗保健服务.csv"],
        "include": r".*",
        "exclude": None,
    },
    "shopping": {
        "theme": "base",
        "label": "购物",
        "files": ["上海市-1754933-utf8.csv-购物服务.csv"],
        "include": r".*",
        "exclude": None,
    },
    "transport": {
        "theme": "base",
        "label": "交通",
        "files": ["上海市-1754933-utf8.csv-交通设施服务.csv"],
        "include": r".*",
        "exclude": None,
    },
    "daily_service": {
        "theme": "base",
        "label": "生活服务",
        "files": ["上海市-1754933-utf8.csv-生活服务.csv", "上海市-1754933-utf8.csv-金融保险服务.csv"],
        "include": r".*",
        "exclude": None,
    },
    "leisure": {
        "theme": "base",
        "label": "休闲",
        "files": ["上海市-1754933-utf8.csv-体育休闲服务.csv", "上海市-1754933-utf8.csv-风景名胜.csv"],
        "include": r".*",
        "exclude": None,
    },
    "public_service": {
        "theme": "base",
        "label": "公共服务",
        "files": ["上海市-1754933-utf8.csv-政府机构及社会团体.csv", "上海市-1754933-utf8.csv-公共设施.csv"],
        "include": r".*",
        "exclude": None,
    },
    "food": {
        "theme": "base",
        "label": "餐饮",
        "files": ["上海市-1754933-utf8.csv-餐饮服务.csv"],
        "include": r".*",
        "exclude": None,
    },
}

TRACK_A_RULES = {
    "fitness": {
        "theme": "track_a",
        "label": "健身房/健身工作室",
        "files": ["上海市-1754933-utf8.csv-体育休闲服务.csv"],
        "include": r"健身|健身中心|健身房|健身工作室|私教|体能训练|普拉提",
        "exclude": r"KTV|酒吧|棋牌|网吧|游戏厅",
    },
    "outdoor_fitness": {
        "theme": "track_a",
        "label": "户外健身器材/健身点",
        "files": ["上海市-1754933-utf8.csv-体育休闲服务.csv", "上海市-1754933-utf8.csv-风景名胜.csv"],
        "include": r"户外健身|健身路径|健身器材|健身点|公园.*健身|健身.*公园",
        "exclude": r"会所|酒店|私人",
    },
    "sport_fields": {
        "theme": "track_a",
        "label": "跑道/运动场/篮球场",
        "files": ["上海市-1754933-utf8.csv-体育休闲服务.csv"],
        "include": r"运动场|运动场所|体育场|综合体育馆|篮球|足球场|羽毛球|乒乓球|网球|跑道|田径",
        "exclude": r"KTV|酒吧|棋牌|网吧|游戏厅",
    },
    "public_pool": {
        "theme": "track_a",
        "label": "公共游泳池",
        "files": ["上海市-1754933-utf8.csv-体育休闲服务.csv", "上海市-1754933-utf8.csv-生活服务.csv"],
        "include": r"游泳馆|游泳池|公共游泳|游泳中心",
        "exclude": r"私人|私家|会所|酒店|宾馆|小区|公寓|别墅|婴儿|亲子|SPA|洗浴",
    },
    "mind_body": {
        "theme": "track_a",
        "label": "瑜伽/武术/舞蹈工作室",
        "files": ["上海市-1754933-utf8.csv-体育休闲服务.csv", "上海市-1754933-utf8.csv-科教文化服务.csv"],
        "include": r"瑜伽|武术|跆拳道|舞蹈|舞室|舞蹈工作室|普拉提|拳击|搏击|太极",
        "exclude": r"KTV|酒吧|棋牌|网吧|游戏厅",
    },
    "fresh_food": {
        "theme": "track_a",
        "label": "新鲜市场/健康食品零售",
        "files": ["上海市-1754933-utf8.csv-购物服务.csv"],
        "include": r"蔬菜市场|农副产品市场|果品市场|水产海鲜市场|菜场|农贸|生鲜|有机|健康食品|超市",
        "exclude": r"烟酒|宠物|家具|建材|五金|数码|服装|鞋|皮具|珠宝|典当",
    },
}

ALL_RULES = BASE_RULES | TRACK_A_RULES
POI_COLUMNS = ["name", "midType", "smallType", "wgs84Lng", "wgs84Lat"]


In [ ]:
def extract_rule_points(rule_key: str, rule: dict) -> pd.DataFrame:
    include = re.compile(rule["include"], re.IGNORECASE)
    exclude = re.compile(rule["exclude"], re.IGNORECASE) if rule.get("exclude") else None
    frames = []
    for filename in rule["files"]:
        path = POI_DIR / filename
        for chunk in pd.read_csv(path, usecols=POI_COLUMNS, chunksize=100_000):
            chunk = chunk.dropna(subset=["wgs84Lng", "wgs84Lat"])
            text = chunk["name"].fillna("") + " " + chunk["midType"].fillna("") + " " + chunk["smallType"].fillna("")
            mask = text.str.contains(include)
            if exclude:
                mask &= ~text.str.contains(exclude)
            filtered = chunk.loc[mask, POI_COLUMNS].copy()
            if not filtered.empty:
                filtered["indicator_key"] = rule_key
                filtered["indicator_label"] = rule["label"]
                filtered["theme"] = rule["theme"]
                frames.append(filtered)
    if not frames:
        return pd.DataFrame(columns=POI_COLUMNS + ["indicator_key", "indicator_label", "theme"])
    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates(subset=["name", "wgs84Lng", "wgs84Lat", "indicator_key"])
    return out

poi_long = pd.concat([extract_rule_points(k, v) for k, v in ALL_RULES.items()], ignore_index=True)
poi_long.head()


In [ ]:
poi_clean_path = CACHE_DIR / "poi_base_tracka_clean.parquet"
poi_long.to_parquet(poi_clean_path, index=False)

poi_summary = (
    poi_long.groupby(["theme", "indicator_key", "indicator_label"])
    .size()
    .reset_index(name="n")
    .sort_values(["theme", "n"], ascending=[True, False])
)
poi_summary


In [ ]:
green_files = sorted(list(AI_DIR.rglob("*AI解译绿地.shp")) + list(AI_DIR.rglob("*Ai解译绿地.shp")))
green_layers = []
for shp in green_files:
    g = gpd.read_file(shp).to_crs("EPSG:4326")
    g["source_file"] = shp.name
    green_layers.append(g[["source_file", "geometry"]])

green_all = gpd.GeoDataFrame(pd.concat(green_layers, ignore_index=True), geometry="geometry", crs="EPSG:4326")
green_cache = CACHE_DIR / "ai_green_all.gpkg"
green_all.to_file(green_cache, layer="green", driver="GPKG")
len(green_all), green_cache


In [ ]:
validation = {
    "poi_clean_path": str(poi_clean_path),
    "boundary_cache": str(boundary_cache),
    "green_cache": str(green_cache),
    "district_count": int(len(districts)),
    "town_count": int(len(towns)),
    "green_polygon_count": int(len(green_all)),
    "poi_counts": poi_summary.to_dict(orient="records"),
    "api_calls": [],
}

with open(CACHE_DIR / "01_validation.json", "w", encoding="utf-8") as f:
    json.dump(validation, f, ensure_ascii=False, indent=2)

validation


## 扩展数据源与验证

本节记录 Healthy Lifestyle 与网页应用使用的扩展数据源，并与前文 POI、行政区边界、绿地数据共同构成后续网格统计和 H3 评分的数据基础。

- `Healthy LifestyleV1.py` 接入上海市生态环境局 2025 年逐日 AQI 数据，优先按起点所在行政区统计年均 AQI、达标天数占比和首要污染物；无法定位行政区时回退到全市均值。
- `sports desert.py` 使用 `data/10-AI解译数据/*建筑.shp` 构造陆地掩膜，只保留与建筑多边形相交的 H3，避免海域和江面被误判为运动荒漠。
- `scripts/build_application_v1_data.py` 整合前序 H3 评分、AQI、最近地铁距离、房价/租金代理，并输出网页端压缩 JSON。


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
SUBMIT_DIR = ROOT / "submit"
CACHE_DIR = SUBMIT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

AQI_DIR = DATA_DIR / "shanghai_aqi_official_2025_20250101_20251231_package" / "clean"
paths = {
    "aqi_city_daily": AQI_DIR / "shanghai_aqi_city_daily_sthj_official_20250101_20251231.csv",
    "aqi_district_daily": AQI_DIR / "shanghai_aqi_district_daily_sthj_official_20250101_20251231.csv",
    "housing_price_csv": DATA_DIR / "06-房价数据" / "上海市-房价数据.csv",
    "metro_station_shp": DATA_DIR / "地铁数据" / "上海市_地铁站.shp",
    "sports_desert_h3": ROOT / "outputs" / "sports_desert" / "sports_desert_h3.parquet",
    "application_v1_compact_json": SUBMIT_DIR / "applicationV1" / "public" / "data" / "app_v1_h3_data_compact.json",
}

ai_building_files = sorted(DATA_DIR.glob("10-AI解译数据/**/*AI解译建筑.shp")) + sorted(DATA_DIR.glob("10-AI解译数据/**/*Ai解译建筑.shp"))

inventory = []
for key, path in paths.items():
    item = {"dataset": key, "path": str(path.relative_to(ROOT)) if path.exists() else str(path), "exists": path.exists()}
    if path.exists() and path.suffix.lower() == ".csv":
        df = pd.read_csv(path, nrows=5)
        item["columns"] = list(df.columns)
    inventory.append(item)

inventory.append({
    "dataset": "ai_building_land_mask_files",
    "path": "data/10-AI解译数据/**/*AI解译建筑.shp",
    "exists": len(ai_building_files) > 0,
    "file_count": len(ai_building_files),
})

summary = {"datasets": inventory}
summary_path = CACHE_DIR / "data_source_validation.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

pd.DataFrame(inventory)


In [ ]:
# AQI 数据质量快速检查：覆盖日期、行政区数量、AQI 均值与达标天数占比。
AQI_DISTRICT_CSV = paths["aqi_district_daily"]
AQI_CITY_CSV = paths["aqi_city_daily"]

if AQI_DISTRICT_CSV.exists():
    aqi_dist = pd.read_csv(AQI_DISTRICT_CSV)
    aqi_dist["date"] = pd.to_datetime(aqi_dist["date"], errors="coerce")
    aqi_dist["aqi"] = pd.to_numeric(aqi_dist["aqi"], errors="coerce")
    district_aqi_summary = (
        aqi_dist.groupby("group_name")
        .agg(
            days=("aqi", "count"),
            first_date=("date", "min"),
            last_date=("date", "max"),
            aqi_avg=("aqi", "mean"),
            good_days_pct=("aqi", lambda s: (s <= 100).mean() * 100),
        )
        .reset_index()
        .sort_values("aqi_avg")
    )
    display(district_aqi_summary.head(10))
else:
    print("未找到区级 AQI CSV：", AQI_DISTRICT_CSV)

if AQI_CITY_CSV.exists():
    aqi_city = pd.read_csv(AQI_CITY_CSV)
    aqi_city["aqi"] = pd.to_numeric(aqi_city["aqi"], errors="coerce")
    print({
        "city_rows": int(len(aqi_city)),
        "city_aqi_avg": round(float(aqi_city["aqi"].mean()), 2),
        "city_good_days_pct": round(float((aqi_city["aqi"] <= 100).mean() * 100), 2),
    })
